# Customer Churn Prediction — End-to-End ML Pipeline
**Author:** Keta Thakkar | University of Cincinnati

## Project Overview
End-to-end machine learning project on customer churn prediction for a national grocery/telecom chain. Covers exploratory data analysis, feature engineering, supervised classification modeling, and an open-ended modeling competition.

## What This Project Covers
- **Part 1: EDA** — Churn rate analysis, feature correlation, noise detection, business insights
- **Part 2: Structured Predictive Modeling** — Logistic Regression, Decision Tree, Random Forest with GridSearchCV and StratifiedKFold cross-validation
- **Part 3: Open-Ended Competition** — Gradient Boosting and HistGradientBoosting with noise removal, achieving ROC AUC of 0.7912

## Key Results
- Best model: **HistGradientBoostingClassifier** (ROC AUC: 0.7912)
- Top churn predictors: Contract type, Internet service, Monthly charges
- Month-to-month customers churned at 16% vs 2-3% for annual contract customers

## Note on Data
The dataset used in this project is proprietary course data from BANA 4080 at the University of Cincinnati and is not included in this repository. The notebook structure, code, analysis, and all written interpretations are entirely my own work.


# **BANA 4080 FINAL PROJECT | KETA THAKKAR**

In [ ]:
# Setup
import pandas as pd
import numpy as np

# models / preprocessing
from sklearn.model_selection import train_test_split, StratifiedKFold, GridSearchCV
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier

from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score


In [ ]:
# Load data
df = pd.read_csv("customer_retention_training.csv")

# Quick peek
df.head()
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 6000 entries, 0 to 5999
Data columns (total 35 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   Status                 6000 non-null   object 
 1   Gender                 6000 non-null   object 
 2   SeniorCitizen          6000 non-null   object 
 3   Partner                6000 non-null   object 
 4   Dependents             6000 non-null   object 
 5   Tenure                 6000 non-null   int64  
 6   Contract               6000 non-null   object 
 7   PaperlessBilling       6000 non-null   object 
 8   PaymentMethod          6000 non-null   object 
 9   MonthlyCharges         6000 non-null   float64
 10  TotalCharges           6000 non-null   float64
 11  PhoneService           6000 non-null   object 
 12  MultipleLines          6000 non-null   object 
 13  InternetService        6000 non-null   object 
 14  OnlineSecurity         6000 non-null   object 
 15  Onli

# **PART 1: Exploratory Data Analysis (EDA)**

###  ***Overall Churn Rate***

In [ ]:
churn_rate = (df['Status'] == 'Left').mean()
print(churn_rate)

0.09916666666666667


9.92% (or 0.09916666666666667) of customers in the dataset have churned

### ***Tenure and Churn***

In [ ]:
df.groupby('Status')['Tenure'].mean()


,Tenure
Status,
Current,36.564107
Left,34.349580


The mean tenure for churned customers is 34.35 months while that for stayed customers is 36.56 months. Hence, churned customers have a slightly lower tenure on average.

### ***Monthly Charges by Contract***

In [ ]:
df.groupby('Contract')['MonthlyCharges'].mean()


,MonthlyCharges
Contract,
Month-to-month,81.520549
One year,77.260549
Two year,71.874439


The mean monthly charge for month-to-month contract customers is 81.52 while that for one year contract customers is 77.26. We see that the average monthly charge drops as the duration of the contract increases.

### ***Churn Rate by Payment Method***

In [ ]:
churn_by_payment = (
    df.groupby('PaymentMethod')['Status']
      .apply(lambda s: (s == 'Left').mean())
      .sort_values(ascending=False)
)
churn_by_payment


,Status
PaymentMethod,
Electronic check,0.132057
Credit card (automatic),0.085202
Bank transfer (automatic),0.082584
Mailed check,0.078032


The churn rates for each of the four payment methods are as follows:
1. Electronic check: 13.21% (or 0.132057)
2. Credit card (automatic): 8.52% (or 0.085202)
3. Bank Transfer (automatic):8.26% (or 0.082584)
4. Mailed Check: 7.8% (or 0.078032)

The churn rates show a clear declining pattern by payment method: electronic checks experience the highest churn, while credit cards, bank transfers, and mailed checks follow in progressively lower order.


### ***App Usage Signal***

In [ ]:
df.groupby('Status')['NumAppLoginsLastMonth'].mean()


,NumAppLoginsLastMonth
Status,
Current,24.607031
Left,23.897479


Customers who churned had mean app logins of 23.90 while those who stayed had mean app logins of 24.61 last month.
So, churned customers logged in slighly less than those who stayed.

### ***Investigating Noise***  

In [ ]:
churn_by_hash = df.groupby('RandomIDHash')['Status'] \
                  .apply(lambda s: (s == 'Left').mean())
churn_by_hash


,Status
RandomIDHash,
X1,0.093586
X2,0.106623
X3,0.102113
X4,0.094682


Churn rates by RandomIDHash are almost identical, suggesting this is just a random identifier and not a meaningful predictor — likely noise.

### ***Dependents and Churn***  

In [ ]:
df.groupby('Dependents')['Status'] \
  .apply(lambda s: (s == 'Left').mean())


,Status
Dependents,
No,0.098462
Yes,0.100845


The churn rate for customers with dependents is 10.08% (or 0.100845) while that for customers without dependents is 9.85% (or 0.098462). It is very similar and, hence, not very predictive.

### ***Email Engagement***  

In [ ]:
df.groupby('Status')['EmailOpenRate'].mean()


,EmailOpenRate
Status,
Current,0.201111
Left,0.196562


The mean email open rate for customers who stayed is 20.11% (or 0.201111) and that for customers who churned is 19.66% (or 0.196562). Since the values are very close, this is also not very predictive.

### ***Churn Rate by Contract Type***  

In [ ]:
churn_by_contract = df.groupby('Contract')['Status'] \
                      .apply(lambda s: (s == 'Left').mean())
churn_by_contract


,Status
Contract,
Month-to-month,0.160243
One year,0.022103
Two year,0.028053


The churn rates for month-to-month, one year, and two year contract customers are 16.02%, 2.21%, and 2.81% respectively. This is a signal since there is a significant increase in churn rate for the month-to-month contract customers.

### ***Tenure Group Comparison***  

In [ ]:
bins = [0, 12, 24, df['Tenure'].max()]
labels = ['<12', '12–24', '>24']

df['TenureGroup'] = pd.cut(df['Tenure'], bins=bins, labels=labels, include_lowest=True)

df.groupby('TenureGroup')['Status'] \
  .apply(lambda s: (s == 'Left').mean())


/tmp/ipython-input-1927137346.py:6: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  df.groupby('TenureGroup')['Status'] \


,Status
TenureGroup,
<12,0.105368
12–24,0.106426
>24,0.095798


The churn rates for customers with tenures <12 months, 12-24 months, and >24 months are 10.54% (or 0.105368), 10.64% (or 0.106426), and 9.58% (0r 0.095798), respectively. They are very similar but the tenure group >24 months has a slightly lower churn rate.

### ***Top Categorical Predictors***  

In [ ]:
categorical_cols = df.select_dtypes(include='object').columns.tolist()
categorical_cols.remove('Status')

cat_diffs = {}

for col in categorical_cols:
    rates = df.groupby(col)['Status'].apply(lambda s: (s == 'Left').mean())
    diff = rates.max() - rates.min()
    cat_diffs[col] = diff

top5_cat = (pd.Series(cat_diffs)
              .sort_values(ascending=False)
              .head(5))
print(top5_cat)


Contract           0.138140
InternetService    0.121428
StreamingMovies    0.088821
StreamingTV        0.086224
TechSupport        0.085021
dtype: float64


The top 5 categorical variables with the largest absolute churn rate difference between the category's levels are: Contract, Internet service, Streaming movies, Streaming TV, and Tech support.

### ***Top Numeric Predictors***  

In [ ]:
numeric_cols = df.select_dtypes(include=['int64', 'float64']).columns.tolist()

num_stats = []

for col in numeric_cols:
    means = df.groupby('Status')[col].mean()
    mean_churn = means['Left']
    mean_stay = means['Current']
    rel_diff = abs((mean_churn - mean_stay) / mean_stay)
    num_stats.append({
        'variable': col,
        'mean_churn': mean_churn,
        'mean_stay': mean_stay,
        'relative_diff': rel_diff
    })

num_stats_df = pd.DataFrame(num_stats)
top5_num = num_stats_df.sort_values('relative_diff', ascending=False).head(5)
top5_num


,variable,mean_churn,mean_stay,relative_diff
1,MonthlyCharges,89.354050,77.318477,0.155662
2,TotalCharges,3097.074168,2823.555920,0.096870
0,Tenure,34.349580,36.564107,0.060566
12,DataOverageCount,1.450420,1.503608,0.035373
5,ReferralCount,4.371429,4.521739,0.033242


The top 5 numeric variables with the largest relative difference are: Monthly charges, Total charges, Tenure, Data overage count, and Referral count.

### ***Data Quality Check***  

In [ ]:
missing = df.isna().sum()
print(missing[missing > 0])


PromoCategory    1469
dtype: int64


PromoCategory has 1469 missing values.

### ***Business Value***  

Identifying noisy features is important because these columns introduce random variation that does not help predict churn. If they remain in the dataset, the model may learn patterns that are purely coincidental, which increases the risk of overfitting and reduces model's generalizability to new customers. Removing noise allows the model to focus on meaningful drivers (signals) of churn, resulting in more stable performance and clearer, more actionable insights for the business.

### ***Most Important Finding***  

One of the strongest signal features in my analysis was **Contract type**. Customers on month-to-month contracts churn at roughly 16.02%, while 1-year and 2-year contract customers churn at only 2.21% and 2.81%. This large gap shows that contract length is a major predictor of churn.

A likely noise feature is **RandomIDHash**. The churn rate across the four hash values was nearly identical (all around ~9–10%), indicating that this column is simply a random identifier and provides no meaningful predictive value.

### ***Feature Correlation***  

Based on my EDA, the two features with the strongest influence on churn are **Contract type** and **MonthlyCharges**. Contract type shows the most significant categorical difference: customers on month-to-month contracts churn at about 16.02%, whereas customers with one-year or two-year contracts churn at only 2.21% and 2.81%. This represents more than a five-fold reduction in churn for customers who commit to longer terms. MonthlyCharges also shows a clear numeric difference. Churned customers have higher average monthly charges (89.35) compared to those who stay (77.32), indicating that higher pricing contributes to customer frustration or perceived lower value.

From a business perspective, these findings suggest that customers who pay more each month and lack long-term commitments are the most vulnerable to leaving. This combination likely reflects customers who feel they are overpaying relative to the value they receive and experience no switching cost if they decide to leave. The insight is actionable: Regork Telecom should design **targeted retention offers that encourage month-to-month customers with high charges** to transition into annual contracts. Potential strategies include **loyalty discounts, bundling services, or introductory reduced rates for contract upgrades**. By addressing price sensitivity and increasing customer commitment, Regork can reduce churn among its highest-risk segment.

##### ***Extra: Monthly Charges and Churn***
Churned customers pay noticeably higher MonthlyCharges on average.
This suggests that customers with higher monthly bills are more likely to leave. This also reinforces my claim that MonthlyCharges is one of the features with the strongest influence on churn.

In [ ]:
df.groupby('Status')['MonthlyCharges'].mean()


,MonthlyCharges
Status,
Current,77.318477
Left,89.354050


##### ***Extra: Churn Rate by Monthly Charge***
We see that as the monthly charge increases, the churn rate also increases significantly. This also reinforces my claim that MonthlyCharges is one of the features with the strongest influence on churn.

In [ ]:
# Create monthly charge buckets
df['ChargeBucket'] = pd.cut(
    df['MonthlyCharges'],
    bins=[0, 50, 70, 90, 120],
    labels=['$0–50', '$50–70', '$70–90', '$90–120'],
    include_lowest=True
)

# Calculate churn rate by bucket
bucket_churn = df.groupby('ChargeBucket')['Status'] \
                 .apply(lambda s: (s == 'Left').mean())

print(bucket_churn)


ChargeBucket
$0–50      0.034161
$50–70     0.046065
$70–90     0.077303
$90–120    0.168188
Name: Status, dtype: float64


/tmp/ipython-input-2720808838.py:10: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  bucket_churn = df.groupby('ChargeBucket')['Status'] \


### ***Limitations***  

While the EDA provides valuable insights into churn behavior, it has several limitations. The first dataset limitation is that the data does not include **customer satisfaction or service quality ratings**, which are typically strong predictors of churn in telecommunications. Without subjective metrics, it is difficult to understand whether customers leave due to dissatisfaction with service, pricing concerns, or other qualitative factors. A second limitation is the absence of competitor information such as competitor prices, promotions, or market changes. Churn decisions often depend on external market pressures, and the dataset cannot capture these influences.

A key analysis limitation is that the EDA is **correlational, not causal**. Although features like Contract and MonthlyCharges show clear relationships with churn, we cannot conclude that they directly cause churn. Other hidden variables or interactions may be influencing the observed patterns. Additionally, **EDA provides a static snapshot** of customer behavior and does not capture changes over time or the sequence of events leading up to churn.

A recommended next step would be to **conduct a survival analysis** to understand when churn risk spikes, using newly collected variables such as satisfaction scores and churn reasons. This would give Regork a deeper and more actionable understanding of how churn risk evolves over time and which factors drive customers to leave.

# **PART 2: Structured Predictive Modeling**



## ***Step 1: Setup***

In [ ]:
X = df.drop(columns=['Status'])
y = df['Status'].map({'Current': 0, 'Left': 1})  # encode as binary

We convert the target variable into a binary numeric form because most machine learning algorithms require numeric inputs for classification tasks. Encoding Left = 1 and Current = 0 allows the model to mathematically learn patterns associated with churn and treat churn as the “positive” class. This also standardizes the evaluation metrics, such as precision and recall, which are defined in terms of predicting the positive class.

## ***Step 2: Train/Test Split***

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.30,
    stratify=y,
    random_state=42
)

X_train.shape, X_test.shape


((4200, 36), (1800, 36))

The training set has 4200 observations.

The test set has 1800 observations.

We use a stratified split to keep the churn rate the same in both the training and test sets. Since only around 9.92% of customers churn, a regular random split might give us a test set with very few churners. By using stratification, we make sure both sets represent the real class balance, which helps the model train properly and makes the test results more reliable.

## ***Step 3: Preprocessing Pipeline***

In [ ]:
# Define numeric & categorical columns

numeric_features = [
    'Tenure',
    'MonthlyCharges',
    'TotalCharges',
    'NumAppLoginsLastMonth',
    'EmailOpenRate',
    'ReferralCount',
    'NumPageViews',
    'AvgCallDuration',
    'NumInternationalCalls',
    'NumAppCrashes',
    'AppSessionLengthAvg',
    'NumPaymentFailures',
    'DataOverageCount'
]

categorical_features = [
    'Gender',
    'SeniorCitizen',
    'Partner',
    'Dependents',
    'Contract',
    'PaperlessBilling',
    'PaymentMethod',
    'PhoneService',
    'MultipleLines',
    'InternetService',
    'OnlineSecurity',
    'OnlineBackup',
    'DeviceProtection',
    'TechSupport',
    'StreamingTV',
    'StreamingMovies',
    'InternalCodeFlag',
    'RandomIDHash',
    'SurveyVersion',
    'LegacySystemScore',
    'PromoCategory'
]

In [67]:
numeric_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='mean')),
    ('scaler', StandardScaler())
])

categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore'))
])

preprocessor = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer, numeric_features),
        ('cat', categorical_transformer, categorical_features)
    ]
)

Putting preprocessing inside a pipeline helps keep all the steps—like imputing missing values, scaling numeric features, and one-hot encoding categorical features—organized and consistent. It also prevents data leakage because the preprocessing is fit only on the training data and then applied to the test data in the same way. Pipelines make the whole modeling process easier to reproduce and deploy. It also allows you to tune preprocessing and model hyperparameters together

## ***Step 4: Cross-Validation***

In [82]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

StratifiedKFold(n_splits=5, random_state=42, shuffle=True)

We use cross-validation to get a more reliable estimate of how well a model will perform on new data. Instead of relying on just one train/test split, cross-validation evaluates the model across several different folds, which reduces the impact of any one split being “lucky” or “unlucky”. This gives us a more stable view of the model’s true performance and helps us choose the best model and tuning parameters.

## ***Step 5: Model Training and Tuning***

#### ***Model A: Logistic Regression***


In [68]:
log_reg = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('clf', LogisticRegression(random_state=42))
])

param_grid_lr = {
    'clf__C': [0.01, 0.1, 1, 10],
    'clf__penalty': ['l2'],
    'clf__solver': ['lbfgs']
}

grid_lr = GridSearchCV(
    log_reg,
    param_grid_lr,
    cv=cv,
    scoring='f1',
    n_jobs=-1
)

grid_lr.fit(X_train, y_train)

print("Best F1 (LogReg):", grid_lr.best_score_)
print("Best params:", grid_lr.best_params_)

Best F1 (LogReg): 0.16904118683353864
Best params: {'clf__C': 10, 'clf__penalty': 'l2', 'clf__solver': 'lbfgs'}


The best cross-validation F1 score for Loistic Regression is 0.16904.

#### ***Model B: Decision Tree***

In [69]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.pipeline import Pipeline
from sklearn.model_selection import GridSearchCV

# Decision Tree pipeline: preprocess + classifier
dt_clf = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('clf', DecisionTreeClassifier(random_state=42))
])

# Hyperparameter grid for Decision Tree
param_grid_dt = {
    'clf__max_depth': [5, 10],
    'clf__min_samples_split': [2, 10],
    'clf__min_samples_leaf': [1, 5]
}

# Grid search with F1 scoring
grid_dt = GridSearchCV(
    estimator=dt_clf,
    param_grid=param_grid_dt,
    cv=cv,
    scoring='f1',
    n_jobs=-1,
    verbose=1
)

grid_dt.fit(X_train, y_train)

print("Best F1 (Decision Tree):", grid_dt.best_score_)
print("Best params:", grid_dt.best_params_)


Fitting 5 folds for each of 8 candidates, totalling 40 fits
Best F1 (Decision Tree): 0.2002625004953164
Best params: {'clf__max_depth': 10, 'clf__min_samples_leaf': 5, 'clf__min_samples_split': 2}


The best cross-validation F1 score for Decision Tree is 0.20026.

#### ***Model C: Random Forest***

In [62]:
rf_clf = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('clf', RandomForestClassifier(random_state=42))
])

param_grid_rf = {
    'clf__n_estimators': [200, 500],
    'clf__max_depth': [5, 10, 20],
    'clf__min_samples_split': [2, 5]
}

grid_rf = GridSearchCV(
    estimator=rf_clf,
    param_grid=param_grid_rf,
    cv=cv,
    scoring='f1',
    n_jobs=-1,
    verbose=1
)

grid_rf.fit(X_train, y_train)

print("Best F1 (Random Forest):", grid_rf.best_score_)
print("Best params:", grid_rf.best_params_)



Fitting 5 folds for each of 12 candidates, totalling 60 fits
Best F1 (Random Forest): 0.014064230343300111
Best params: {'clf__max_depth': 20, 'clf__min_samples_split': 2, 'clf__n_estimators': 200}


The best cross-validation F1 score for Random Forest is 0.01406.

## ***Step 6: Model Selection***

In [ ]:
cv_scores = {
    'Logistic Regression': grid_lr.best_score_,
    'Decision Tree': grid_dt.best_score_,
    'Random Forest': grid_rf.best_score_
}
cv_scores

{'Logistic Regression': np.float64(0.16904118683353864),
 'Decision Tree': np.float64(0.2002625004953164),
 'Random Forest': np.float64(0.014064230343300111)}

I chose the Decision Tree as my best model because it had the highest cross-validated F1 score (0.20026) compared to Logistic Regression (0.16904) and Random Forest (0.01406).

## ***Step 7: Feature Importance***

In [74]:
from sklearn.inspection import permutation_importance
import pandas as pd
import numpy as np

best_model = grid_dt.best_estimator_

# Calculate feature importance
result = permutation_importance(
    best_model,
    X_test,
    y_test,
    n_repeats=10,
    random_state=42,
    n_jobs=-1,
    scoring='f1'
)

# The feature importances returned by permutation_importance correspond to the original features in X_test
feature_names = X_test.columns

importances = pd.DataFrame({
    'feature': feature_names,
    'importance_mean': result.importances_mean,
    'importance_std': result.importances_std
})

importances = importances.sort_values(by='importance_mean', ascending=False)
top_5_features_df = importances.head(5)

print("Top 5 most important features:")
for i, row in top_5_features_df.iterrows():
    print(f"{i+1}. {row['feature']} (Mean Importance: {row['importance_mean']:.4f})")

Top 5 most important features:
13. InternetService (Mean Importance: 0.0456)
6. Contract (Mean Importance: 0.0410)
28. NumPaymentFailures (Mean Importance: 0.0164)
8. PaymentMethod (Mean Importance: 0.0147)
31. RandomIDHash (Mean Importance: 0.0068)


According to permutation importance, the top 5 most important features are: InternetService, Contract, NumPaymentFailures, PaymentMethod, and RandomIDHash

## ***Step 8: Evaluation***

In [77]:
y_pred = best_model.predict(X_test)

accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred)
recall = recall_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)

print("Accuracy:", accuracy)
print("Precision:", precision)
print("Recall:", recall)
print("F1:", f1)

Accuracy: 0.8694444444444445
Precision: 0.1724137931034483
Recall: 0.08426966292134831
F1: 0.11320754716981132


The accuracy of the best model is 86.94%.

The precision of the best model is 17.24%.


The recall of the best model is 8.43%.


The F1 of the best model is 11.32%.


## ***Step 9: Interpretation***

The best model had a cross-validation F1 score of 0.2003, but its test set F1 score dropped to 0.1132, a difference of about 0.087. This drop shows some (moderate) overfitting because the model performed much better during cross-validation than on new data. It likely picked up patterns that only existed in the training folds, which is common with decision trees and small churn classes.

Generalization means how well a model works on new customers it hasn’t seen before. The recall is only 8.43%, meaning that out of every 100 customers who are going to churn, the model only flags about 8 of them. Relying on this would cause us to miss around 92% of customers we’re about to lose, making it unsuitable as an early warning system. A major concern is that the model may give the business a false sense of confidence by predicting almost everyone as staying, which makes it unreliable for real retention decisions.

### ***Feature Interpretation***  

The top two features identified by my model were InternetService and Contract.Contract type showed the strongest signal in the dataset: month-to-month customers churned at 16.02%, while one-year and two-year contract customers churned at only 2.21% and 2.81%. This represents a seven-fold difference, making contract commitment one of the clearest predictors of churn. InternetService reflects whether a customer uses DSL, Fiber, or no internet service. Our EDA showed meaningful differences between groups, with Fiber customers churning at 15.47% and DSL users churning at 4.96%- which is a significant difference- suggesting that customers paying for premium speed may be more sensitive to performance or pricing issues. However, many other features in this dataset, such as tenure (2.2 months difference) and engagement metrics (0.7 logins difference), showed much weaker signals than typical telecom datasets.This suggests our model must rely heavily on just a few strong predictors while the rest contribute minimally.

A specific recommendation is to target month-to-month Fiber customers, the highest-risk group. Offering a 20% contract-conversion discount or a 90-day proactive service check-in could address both price sensitivity and service expectations. This ties model insights directly to actionable retention strategy.

##### ***Extra: Churn Rate by Internet Service***

In [81]:
# Calculate churn rate by InternetService
df.groupby("InternetService")["Status"].apply(lambda x: (x == "Left").mean())


,Status
InternetService,
DSL,0.049618
Fiber optic,0.154726
No,0.033298


### ***Model Recommendation***  

Among the three models tested—Logistic Regression, Decision Tree, and Random Forest—the Decision Tree is the most defensible choice, even though overall performance remains limited. The Decision Tree achieved the highest cross-validated F1 score at 0.2003, outperforming Logistic Regression (0.1690) and Random Forest (0.01406). On the test set, the Decision Tree’s F1 dropped to 0.1132, with a precision of 17.24% and recall of only 8.43%, meaning it identifies fewer than 9 out of every 100 customers who churn. Hence, the high accuracy of around 87% is misleading because because the model is essentially achieving that score by simply predicting that every customer will stay, which happens 90% of the time anyway.

Random Forest failed almost completely, producing near-zero F1 scores on both cross-validation and test set, so it cannot be considered a viable model.

In generalization, none of the models performed well. The Decision Tree shows a noticeable drop from cross-validation to the test, indicating overfitting, while Logistic Regression generalizes slightly more consistently but still fails to capture meaningful churn patterns. Only 8.43% recall of best model means they miss over 90% of customers who actually churn. This makes accuracy misleading and highlights that these models are not ready for operational use.

Although the Decision Tree is the strongest of the three, this is a “least bad” recommendation. It should be used only as a baseline for further improvement, not for real churn prediction. Regork would need a stronger modeling approach or richer data before deploying this into production.

# **PART 3: Open-Ended Modeling**


### ***TRY 1***

Given this dataset (tabular, mixed numeric + categorical, modest size, slight imbalance), I think a Gradient Boosting model is generally stronger than Logistic Regression, Decision Tree, or Random Forest for ROC AUC- I also saw how Logistic Regression, Decision Tree, and Random Forest performed in Part 2, so I think Gradient Boosting model is the way to go.


We’ll remove at least the obvious noise: RandomIDHash
We’ll keep everything else because they may still carry weak signal.
So,
Training X: all columns except Status and RandomIDHash
Test X (from test_features.csv): all columns except RandomIDHash

In [84]:
import pandas as pd
from sklearn.model_selection import StratifiedKFold, GridSearchCV, train_test_split
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.metrics import roc_auc_score

# 1. Load data

train_df = pd.read_csv("customer_retention_training.csv")
test_df  = pd.read_csv("test_features.csv")

# Encode target: Current -> 0, Left -> 1
y = train_df["Status"].map({"Current": 0, "Left": 1})

# Drop target and noise column from training features
X = train_df.drop(columns=["Status", "RandomIDHash"])

# Drop noise column from test features
X_test_final = test_df.drop(columns=["RandomIDHash"])


In [85]:
# 2. Preprocessing

numeric_features = X.select_dtypes(include=["int64", "float64"]).columns.tolist()
categorical_features = X.select_dtypes(include=["object"]).columns.tolist()

numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="mean")),
    ("scaler", StandardScaler())
])

categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numeric_features),
        ("cat", categorical_transformer, categorical_features)
    ]
)


In [86]:
# 3. Gradient Boosting model + ROC AUC tuning

gb_clf = GradientBoostingClassifier(random_state=42)

pipe_gb = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("clf", gb_clf)
])

param_grid_gb = {
    "clf__n_estimators": [200, 400],
    "clf__learning_rate": [0.05, 0.1],
    "clf__max_depth": [3, 4]
}

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

grid_gb = GridSearchCV(
    estimator=pipe_gb,
    param_grid=param_grid_gb,
    cv=cv,
    scoring="roc_auc",
    n_jobs=-1,
    verbose=1
)

# Fit on all available training data
grid_gb.fit(X, y)

print("Best ROC AUC (CV):", grid_gb.best_score_)
print("Best params:", grid_gb.best_params_)

Fitting 5 folds for each of 8 candidates, totalling 40 fits
Best ROC AUC (CV): 0.790424365861053
Best params: {'clf__learning_rate': 0.05, 'clf__max_depth': 3, 'clf__n_estimators': 200}


In [88]:
best_model = grid_gb.best_estimator_

In [90]:
# quick check

X_train_sub, X_val_sub, y_train_sub, y_val_sub = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

best_model.fit(X_train_sub, y_train_sub)
val_probs = best_model.predict_proba(X_val_sub)[:, 1]
print("Validation ROC AUC:", roc_auc_score(y_val_sub, val_probs))


Validation ROC AUC: 0.7556883993190244


I just want to remove other sources of noise (which I assumed would have minor contributions) and then select whichever model has the better ROC AUC.

### ***TRY 2***

To improve my model, I removed several “noise” features—columns that don’t actually tell us anything about why customers churn. Noise can come from things like ID codes, survey versions, or internal tags that have nothing to do with customer behavior. In this dataset, RandomIDHash, SurveyVersion, InternalCodeFlag, PromoCategory, and LegacySystemScore were mainly administrative fields, so I removed them. Keeping these would just confuse the model and cause overfitting. After dropping them, the model focused on real customer patterns and achieved a better ROC AUC.

In [91]:
import pandas as pd
from sklearn.model_selection import StratifiedKFold, GridSearchCV
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.experimental import enable_hist_gradient_boosting
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import roc_auc_score

# LOAD DATA

train_df = pd.read_csv("customer_retention_training.csv")
test_df = pd.read_csv("test_features.csv")

y = train_df["Status"].map({"Current":0, "Left":1})

# Remove noise columns
noise_cols = ["RandomIDHash", "SurveyVersion", "InternalCodeFlag",
              "PromoCategory", "LegacySystemScore"]

X = train_df.drop(columns=noise_cols + ["Status"])
X_test_final = test_df.drop(columns=noise_cols)



/usr/local/lib/python3.12/dist-packages/sklearn/experimental/enable_hist_gradient_boosting.py:19: UserWarning: Since version 1.0, it is not needed to import enable_hist_gradient_boosting anymore. HistGradientBoostingClassifier and HistGradientBoostingRegressor are now stable and can be normally imported from sklearn.ensemble.
  warnings.warn(


In [92]:
# PREPROCESSING

numeric = X.select_dtypes(include=["int64","float64"]).columns.tolist()
categorical = X.select_dtypes(include=["object"]).columns.tolist()

numeric_transformer = Pipeline([
    ("imp", SimpleImputer(strategy="mean")),
    ("sc", StandardScaler())
])

categorical_transformer = Pipeline([
    ("imp", SimpleImputer(strategy="most_frequent")),
    ("oh", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer([
    ("num", numeric_transformer, numeric),
    ("cat", categorical_transformer, categorical)
])



In [93]:
# BEST MODEL: HGB

hgb = HistGradientBoostingClassifier(random_state=42)

pipe = Pipeline([
    ("pre", preprocessor),
    ("clf", hgb)
])

param_grid = {
    "clf__learning_rate": [0.05, 0.1],
    "clf__max_depth": [3, 4, 6],
    "clf__max_leaf_nodes": [15, 31],
    "clf__min_samples_leaf": [20, 30, 50]
}

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

grid = GridSearchCV(
    pipe,
    param_grid,
    scoring="roc_auc",
    cv=cv,
    n_jobs=-1,
    verbose=1
)


In [94]:
# Train on full dataset
grid.fit(X, y)

print("Best ROC AUC:", grid.best_score_)
print("Best params:", grid.best_params_)

# GENERATE FINAL PREDICTIONS

best_model = grid.best_estimator_

test_probs = best_model.predict_proba(X_test_final)[:,1]

prediction_df = pd.DataFrame({
    "ChurnProbability": test_probs
})

prediction_df.to_csv("part3_predictions.csv", index=False)
print("Saved part3_predictions.csv")


Fitting 5 folds for each of 36 candidates, totalling 180 fits
Best ROC AUC: 0.791154315565264
Best params: {'clf__learning_rate': 0.05, 'clf__max_depth': 3, 'clf__max_leaf_nodes': 15, 'clf__min_samples_leaf': 30}
Saved part3_predictions.csv


As we can see, the ROC AUC improved from 0.7557 to 0.7912. So, we are going to use Try 2- High Gradient Boosting with lesser noise!